<a href="https://colab.research.google.com/github/Ariqq16/ML_2026/blob/main/JS04/Tugas_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("/content/sample_data/insurance.csv")
print(df.head())
print(df.info())

   age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB
None


In [4]:
# variabel fitur
X = df.drop("charges", axis=1)

# variabel target
y = df["charges"]

print("Variabel bebas:")
print(X.columns.tolist())

print("\nVariabel target:")
print(y.name)

Variabel bebas:
['age', 'sex', 'bmi', 'children', 'smoker', 'region']

Variabel target:
charges


In [5]:
# membagi dataset menjadi data latih dan data uji
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Jumlah data keseluruhan :", len(df))
print("Jumlah data training    :", len(X_train))
print("Jumlah data testing     :", len(X_test))

Jumlah data keseluruhan : 1338
Jumlah data training    : 1070
Jumlah data testing     : 268


In [9]:
# feature scaling
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features = [
    "age",
    "bmi",
    "children"
]

categorical_features = [
    "sex",
    "smoker",
    "region"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

print("Preprocessing berhasil dibuat.")

Preprocessing berhasil dibuat.


In [10]:
# multiple linear regression
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

print("Model Multiple Linear Regression berhasil dibuat.")

Model Multiple Linear Regression berhasil dibuat.


In [12]:
# melatih model
linear_model.fit(X_train, y_train)

print("Model berhasil dilatih.")

Model berhasil dilatih.


In [13]:
# prediksi
y_pred_linear = linear_model.predict(X_test)

print("Hasil prediksi:")
print(y_pred_linear[:10])

Hasil prediksi:
[ 8969.55027444  7068.74744287 36858.41091155  9454.67850053
 26973.17345656 10864.11316424   170.28084136 16903.45028662
  1092.43093614 11218.34318352]


In [14]:
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

# R-squared
r2_linear = r2_score(
    y_test,
    y_pred_linear
)

# MSE
mse_linear = mean_squared_error(
    y_test,
    y_pred_linear
)

# MAE
mae_linear = mean_absolute_error(
    y_test,
    y_pred_linear
)

print("=== Evaluasi Multiple Linear Regression ===")
print("R-squared :", r2_linear)
print("MSE       :", mse_linear)
print("MAE       :", mae_linear)

=== Evaluasi Multiple Linear Regression ===
R-squared : 0.7835929767120723
MSE       : 33596915.85136146
MAE       : 4181.194473753649


In [15]:
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV

In [16]:
svr_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", SVR())
    ]
)

print("Pipeline SVR berhasil dibuat.")

Pipeline SVR berhasil dibuat.


In [17]:
param_grid = {
    "model__kernel": [
        "rbf",
        "linear"
    ],

    "model__C": [
        1,
        10,
        100,
        1000
    ],

    "model__epsilon": [
        0.1,
        0.5,
        1
    ],

    "model__gamma": [
        "scale",
        "auto"
    ]
}

In [18]:
grid_search = GridSearchCV(
    estimator=svr_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Hyperparameter tuning selesai.")

Hyperparameter tuning selesai.


In [19]:
print("Parameter terbaik:")
print(grid_search.best_params_)

Parameter terbaik:
{'model__C': 1000, 'model__epsilon': 0.1, 'model__gamma': 'scale', 'model__kernel': 'linear'}


In [20]:
# prediksi SVR
y_pred_svr = grid_search.predict(X_test)

print("Hasil prediksi SVR:")
print(y_pred_svr[:10])

Hasil prediksi SVR:
[ 9305.11547045  6093.4096054  33322.3789939   9223.09109398
 21254.43269661  5699.54680611  1284.97589938 13363.83084731
  3720.82318677 10143.88239259]


In [21]:
# R-squared
r2_svr = r2_score(
    y_test,
    y_pred_svr
)

# MSE
mse_svr = mean_squared_error(
    y_test,
    y_pred_svr
)

# MAE
mae_svr = mean_absolute_error(
    y_test,
    y_pred_svr
)

print("=== Evaluasi SVR ===")
print("R-squared :", r2_svr)
print("MSE       :", mse_svr)
print("MAE       :", mae_svr)

=== Evaluasi SVR ===
R-squared : 0.7170402426668855
MSE       : 43929143.38918608
MAE       : 3458.736526392645
